In [2]:
import sys

# 1. Tenta importar o ipywidgets (se falhar, instala)
try:
    import ipywidgets as widgets
except ImportError:
    print("📦 Instalando 'ipywidgets' para a interface gráfica...")
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    import ipywidgets as widgets

# 2. Tenta importar o pyperclip (se falhar, instala)
try:
    import pyperclip
except ImportError:
    print("📦 Instalando 'pyperclip' para o Ctrl+C automático...")
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyperclip"])
    import pyperclip

# 3. Bibliotecas nativas do Python
import inspect
from datetime import datetime
from pathlib import Path
from IPython.display import HTML, clear_output, display

print("✅ Todas as dependências carregadas com sucesso!")


# --- 1. Sua função de leitura com Filtros ---
def get_prompt_base():
    try:
        current_dir = Path(__file__).resolve().parent
    except NameError:
        current_dir = Path.cwd()

    src_dir = current_dir / "src"

    header_text = inspect.cleandoc("""
        Você é um desenvolvedor especialista em jogos e arquitetura da Phaser (Phaser 3).
        Abaixo estão os arquivos do meu projeto de um JOGO 2D estruturados em Markdown.
        Analise-os como arquivos individuais dentro da árvore de diretórios especificada para responder às minhas próximas perguntas.

        DIRETRIZES OBRIGATÓRIAS PARA SUAS RESPOSTAS:
        1. PADRÕES PHASER: Todas as alterações, refatorações ou novos códigos sugeridos devem seguir estritamente as melhores práticas e padrões de projeto do Phaser.
        2. CÓDIGO COMPLETO: Sempre que eu solicitar uma alteração ou correção em um arquivo, NÃO envie apenas trechos. Você DEVE fornecer o código do arquivo modificado COMPLETAMENTE.
    """)

    parts = [header_text, "---"]
    PATH_IGNORE_LIST = {"node_modules", "dist", ".git", "libs", "vendor"}

    try:
        if not src_dir.exists():
            return "Erro: Pasta 'src' não encontrada."

        files = sorted(src_dir.rglob("*.js"))
        read_files_count = 0

        for file_path in files:
            if file_path.is_file():
                path_parts = set(file_path.parts)
                if (
                    file_path.name.endswith(".min.js")
                    or not path_parts.isdisjoint(PATH_IGNORE_LIST)
                ):
                    continue

                relative_path = file_path.relative_to(src_dir)
                parts.append(f"# FILE: src/{relative_path}\n```javascript")
                with open(file_path, "r", encoding="utf-8") as f:
                    parts.append(f.read())
                parts.append("```")
                read_files_count += 1

        if read_files_count == 0:
            return "Nenhum arquivo .js válido encontrado na pasta src."

        return "\n\n".join(parts)
    except Exception as e:
        return f"Erro ao ler arquivos: {e}"


# --- 2. Injeção de CSS para o Tema VS Code Dark ---
vscode_style = widgets.HTML("""
<style>
    .widget-textarea textarea {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
        border: 1px solid #3c3c3c !important;
        font-family: 'Consolas', 'Fira Code', monospace !important;
        font-size: 14px !important;
        border-radius: 4px !important;
        padding: 10px !important;
    }
    .widget-textarea textarea:focus {
        border-color: #007acc !important;
    }
    .widget-label {
        color: #858585 !important;
        font-weight: bold !important;
    }
    .vscode-output-box {
        background-color: #1e1e1e;
        color: #d4d4d4;
        font-family: 'Consolas', 'Fira Code', monospace;
        font-size: 13px;
        padding: 15px;
        border-radius: 4px;
        border: 1px solid #3c3c3c;
        white-space: pre-wrap;
        max-height: 400px;
        overflow-y: auto;
    }
</style>
""")

# --- 3. Criação dos Componentes Visuais ---
input_user_request = widgets.Textarea(
    value="",
    placeholder="Ex: Ajuste a física do pulo no PlayScene.js para ficar mais fluido...",
    description="O que fazer:",
    layout=widgets.Layout(width="100%", height="150px"),
    disabled=False,
)

btn_generate = widgets.Button(
    description="Gerar Prompt Final",
    tooltip="Clique para juntar o código, salvar o histórico e copiar para o seu Ctrl+C",
    icon="copy",
)
btn_generate.style.button_color = "#0e639c"
btn_generate.style.text_color = "#ffffff"

output_area = widgets.Output()


# --- 4. Função de clique com Salvamento e Cópia Automática ---
def handle_click(b):
    with output_area:
        clear_output()

        user_request = input_user_request.value.strip()

        if not user_request:
            display(
                HTML(
                    "<span style='color: #f48771; font-weight: bold;'>❌ Por favor, digite o que você deseja alterar antes de gerar!</span>"
                )
            )
            return

        prompt_base = get_prompt_base()

        final_prompt = inspect.cleandoc(f"""
            {prompt_base}
            
            ---
            
            ### SOLICITAÇÃO ATUAL DO DESENVOLVEDOR (EXECUTE AGORA):
            {user_request}
        """)

        char_count = len(final_prompt)

        # 1. Salva no Histórico (.txt)
        try:
            history_dir = Path.cwd() / ".prompt_history"
            history_dir.mkdir(exist_ok=True)

            timestamp = datetime.now().strftime("%d-%m-%Y_%H%M")
            file_name = f"prompt_{timestamp}.txt"
            file_path = history_dir / file_name

            file_path.write_text(final_prompt, encoding="utf-8")
            history_status = f"💾 Arquivo saved em: <span style='color: #b5cea8;'>.prompt_history/{file_name}</span>"
        except Exception as e:
            history_status = f"⚠️ Erro ao salvar arquivo: {e}"

        # 2. Executa a cópia automática para a Área de Transferência (Ctrl + C)
        try:
            pyperclip.copy(final_prompt)
            copy_status = "<span style='color: #4ec9b0; font-weight: bold;'>📋 COPIADO PARA A ÁREA DE TRANSFERÊNCIA!</span>"
        except Exception as e:
            copy_status = f"<span style='color: #f48771;'>⚠️ Não foi possível copiar automaticamente: {e}</span>"

        # Exibe o painel de sucesso atualizado
        result_html = f"""
        <div style='color: #4fc1ff; font-weight: bold; font-size: 15px; margin-bottom: 5px;'>{copy_status}</div>
        <div style='font-size: 12px; color: #858585; margin-bottom: 10px;'>
            {history_status} | 📊 Tamanho: <span style='color: #ce9178;'>{char_count:,} caracteres</span>
        </div>
        <div style='font-size: 11px; color: #858585; margin-bottom: 2px;'>Visualização do conteúdo copiado:</div>
        <div class='vscode-output-box'>{final_prompt}</div>
        """
        display(HTML(result_html))


btn_generate.on_click(handle_click)

# --- 5. Exibe tudo na tela ---
display(vscode_style)
display(input_user_request)
display(btn_generate)
display(output_area)

✅ Todas as dependências carregadas com sucesso!


HTML(value="\n<style>\n    .widget-textarea textarea {\n        background-color: #1e1e1e !important;\n       …

Textarea(value='', description='O que fazer:', layout=Layout(height='150px', width='100%'), placeholder='Ex: A…

Button(description='Gerar Prompt Final', icon='copy', style=ButtonStyle(button_color='#0e639c', text_color='#f…

Output()